# File-level Streaming Pipeline

Pipeline này tránh full disk bằng cách:
1. tải từng zip,
2. mở zip bằng Python,
3. extract từng `.parquet` một file tạm,
4. Spark đọc/lọc/append,
5. xóa file tạm ngay.

In [1]:
!pip -q install pyspark tqdm

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("telegram-file-level-stream")
    .config("spark.sql.shuffle.partitions", "32")
    .config("spark.driver.memory", "8g")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 00:23:18 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
from pathlib import Path
import gc, shutil, subprocess, zipfile
from tqdm.auto import tqdm

from pyspark.sql.functions import col, length, to_timestamp, log1p
from pyspark.sql.functions import avg, max as spark_max, count

HF_BASE = "https://huggingface.co/datasets/Tungtom2004/Telegram_politic_dataset/resolve/main"

BASE = Path("/kaggle/working/data")
RAW = BASE / "raw_tmp"
TMP = BASE / "parquet_tmp"
FINAL = BASE / "final" / "toxic_political_amplified_filelevel"

RAW.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)
FINAL.parent.mkdir(parents=True, exist_ok=True)

ZIP_FILES = [
    "channels_10_parquet.zip",
    "channels_11_parquet.zip",
    "channels_12_parquet.zip",
    "channels_13_parquet.zip",
    "channels_14_parquet.zip",
    "channels_15_parquet.zip",
    "channels_16_parquet.zip",
    "channels_17_parquet.zip",
    "channels_18_parquet.zip",
    "channels_19_parquet.zip",
    "channels_20_parquet.zip",
    "channels_21_parquet.zip",
    "channels_22_parquet.zip",
    "channels_23_parquet.zip",
    "channels_24_parquet.zip",
]

KEEP_COLS = [
    "content", "language", "date", "political",
    "toxicity", "severe_toxicity", "identity_attack",
    "insult", "profanity", "threat", "forwards",
]

RESET_OUTPUT = False

MIN_CONTENT_LENGTH = 50
MIN_FORWARDS = 1
MIN_TOXICITY = 0.2

SPAM_REGEX = "(?i)(http|www|vip|deposit|register|bonus|usdt|airdrop|referral|withdrawal|claim|earn)"

In [4]:
def free_gb(path="/kaggle/working"):
    return shutil.disk_usage(path).free / 1e9

def run_quiet(cmd):
    result = subprocess.run(cmd, stdout=subprocess.DEVNULL, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr[:2000])

def safe_remove_file(path):
    path = Path(path)
    if path.exists():
        path.unlink()

def safe_remove_dir(path):
    path = Path(path)
    if path.exists():
        shutil.rmtree(path, ignore_errors=True)

def cleanup_temp():
    safe_remove_dir(TMP)
    TMP.mkdir(parents=True, exist_ok=True)
    spark.catalog.clearCache()
    gc.collect()

def download_zip(zip_name):
    zip_path = RAW / zip_name
    safe_remove_file(zip_path)
    run_quiet(["wget", "-O", str(zip_path), f"{HF_BASE}/{zip_name}"])
    return zip_path

if RESET_OUTPUT and FINAL.exists():
    shutil.rmtree(FINAL, ignore_errors=True)

safe_remove_dir(RAW)
safe_remove_dir(TMP)
RAW.mkdir(parents=True, exist_ok=True)
TMP.mkdir(parents=True, exist_ok=True)

print(f"Output path: {FINAL}")
print(f"Free disk: {free_gb():.2f} GB")
print(f"Thresholds: toxicity >= {MIN_TOXICITY}, forwards >= {MIN_FORWARDS}, length >= {MIN_CONTENT_LENGTH}")

Output path: /kaggle/working/data/final/toxic_political_amplified_filelevel
Free disk: 20.94 GB
Thresholds: toxicity >= 0.2, forwards >= 1, length >= 50


In [5]:
def filter_and_append_parquet(parquet_path):
    df = spark.read.parquet(str(parquet_path))

    existing_cols = [c for c in KEEP_COLS if c in df.columns]
    required = ["content", "language", "political", "toxicity", "forwards"]
    missing = [c for c in required if c not in existing_cols]
    if missing:
        return 0, f"missing columns {missing}"

    df = df.select(*existing_cols)

    # political trong parquet thực tế là BIGINT 0/1, nên dùng == 1
    df = (
        df
        .filter(col("content").isNotNull())
        .filter(length(col("content")) >= MIN_CONTENT_LENGTH)
        .filter(col("language") == "en")
        .filter(col("political") == 1)
        .filter(col("toxicity").isNotNull())
        .filter(col("toxicity") >= MIN_TOXICITY)
        .filter(col("forwards").isNotNull())
        .filter(col("forwards") >= MIN_FORWARDS)
        .filter(~col("content").rlike(SPAM_REGEX))
    )

    if "date" in existing_cols:
        df = (
            df.withColumn("date_ts", to_timestamp("date"))
              .filter(col("date_ts").isNotNull())
              .drop("date")
        )

    df = df.withColumn("amplification_score", log1p(col("forwards")))

    n = df.count()
    if n > 0:
        df.write.mode("append").parquet(str(FINAL))

    del df
    spark.catalog.clearCache()
    gc.collect()
    return n, None

In [6]:
total_rows = 0
failed_items = []

zip_progress = tqdm(ZIP_FILES, desc="ZIP files", unit="zip")

for zip_name in zip_progress:
    zip_path = None

    try:
        if free_gb() < 4:
            raise RuntimeError(f"Low disk before download: {free_gb():.2f} GB")

        zip_progress.set_postfix_str(f"download {zip_name}")
        zip_path = download_zip(zip_name)

        with zipfile.ZipFile(zip_path, "r") as z:
            parquet_names = [n for n in z.namelist() if n.endswith(".parquet")]

            inner_progress = tqdm(
                parquet_names,
                desc=zip_name,
                unit="parquet",
                leave=False
            )

            for idx, member in enumerate(inner_progress):
                tmp_file = TMP / f"tmp_{idx}.parquet"

                try:
                    if free_gb() < 2:
                        raise RuntimeError(f"Low disk during extraction: {free_gb():.2f} GB")

                    with z.open(member) as src, open(tmp_file, "wb") as dst:
                        shutil.copyfileobj(src, dst, length=1024 * 1024 * 16)

                    n, err = filter_and_append_parquet(tmp_file)
                    total_rows += n

                    safe_remove_file(tmp_file)
                    inner_progress.set_postfix_str(f"rows={total_rows:,} free={free_gb():.1f}GB")

                    if err:
                        failed_items.append((zip_name, member, err))

                except Exception as e:
                    failed_items.append((zip_name, member, str(e)[:300]))
                    safe_remove_file(tmp_file)
                    spark.catalog.clearCache()
                    gc.collect()
                    continue

            inner_progress.close()

        safe_remove_file(zip_path)
        cleanup_temp()
        zip_progress.set_postfix_str(f"rows={total_rows:,} free={free_gb():.1f}GB")

    except Exception as e:
        failed_items.append((zip_name, "__zip_level__", str(e)[:300]))
        if zip_path is not None:
            safe_remove_file(zip_path)
        cleanup_temp()
        zip_progress.set_postfix_str(f"failed {zip_name}")
        continue

zip_progress.close()

print(f"Done. Total rows appended: {total_rows:,}")
print(f"Output path: {FINAL}")
print(f"Free disk: {free_gb():.2f} GB")

if failed_items:
    print(f"Failed items: {len(failed_items)}")
    for item in failed_items[:20]:
        print(item)

ZIP files:   0%|          | 0/15 [00:00<?, ?zip/s]

channels_10_parquet.zip:   0%|          | 0/810 [00:00<?, ?parquet/s]

channels_11_parquet.zip:   0%|          | 0/2418 [00:00<?, ?parquet/s]

channels_12_parquet.zip:   0%|          | 0/3203 [00:00<?, ?parquet/s]

channels_13_parquet.zip:   0%|          | 0/3231 [00:00<?, ?parquet/s]

...

channels_21_parquet.zip:   0%|          | 0/3966 [00:00<?, ?parquet/s]

channels_22_parquet.zip:   0%|          | 0/1536 [00:00<?, ?parquet/s]

channels_23_parquet.zip:   0%|          | 0/49 [00:00<?, ?parquet/s]

channels_24_parquet.zip:   0%|          | 0/36 [00:00<?, ?parquet/s]

Done. Total rows appended: 1,732,259
Output path: /kaggle/working/data/final/toxic_political_amplified_filelevel
Free disk: 20.37 GB
Failed items: 67
('channels_15_parquet.zip', 'channels_15_parquet/channel_1502533790.parquet', 'An error occurred while calling o1101465.parquet.\n: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 57394.0 failed 1 times, most recent failure: Lost task 0.0 in stage 57394.0 (TID 48560) (7ab9cc1a2daa executor driver): org.apache.spark.SparkException: Exception th')
('channels_18_parquet.zip', 'channel_1872306404.parquet', 'An error occurred while calling o2077852.parquet.\n: org.apache.spark.SparkException: Job aborted due to stage failure: Task 0 in stage 108282.0 failed 1 times, most recent failure: Lost task 0.0 in stage 108282.0 (TID 91531) (7ab9cc1a2daa executor driver): org.apache.spark.SparkException: Exception ')
('channels_18_parquet.zip', 'channel_1872375289.parquet', 'An error occurred while calling o2090737.parq

In [7]:
if not FINAL.exists():
    print("FINAL output does not exist yet. No rows passed the filters.")
else:
    parquet_files = list(FINAL.rglob("*.parquet"))
    print("Output parquet files:", len(parquet_files))

    if not parquet_files:
        print("FINAL exists but has no parquet files.")
    else:
        df_final = spark.read.parquet(str(FINAL))
        print("Rows:", df_final.count())
        df_final.show(5, truncate=120)

Output parquet files: 11854


Rows: 1732259
+------------------------------------------------------------------------------------------------------------------------+--------+---------+--------+---------------+---------------+------+---------+------+--------+-------------------+-------------------+
|                                                                                                                 content|language|political|toxicity|severe_toxicity|identity_attack|insult|profanity|threat|forwards|            date_ts|amplification_score|
+------------------------------------------------------------------------------------------------------------------------+--------+---------+--------+---------------+---------------+------+---------+------+--------+-------------------+-------------------+
|BREAKING: The usurping Zionist entity has just cut off the internet to Gaza again. It's a total telecommunications bl...|      en|        1|    0.63|           0.35|           0.63|  0.51|     0.38|  0.52|      12|202

In [8]:
if FINAL.exists() and list(FINAL.rglob("*.parquet")):
    df_final = spark.read.parquet(str(FINAL))

    df_final.agg(
        count("*").alias("n_rows"),
        avg("toxicity").alias("avg_toxicity"),
        avg("insult").alias("avg_insult"),
        avg("threat").alias("avg_threat"),
        avg("forwards").alias("avg_forwards"),
        spark_max("forwards").alias("max_forwards")
    ).show(truncate=False)

    df_final.orderBy(col("forwards").desc()).select(
        "content", "toxicity", "insult", "threat",
        "forwards", "amplification_score"
    ).show(20, truncate=160)

+-------+------------------+-------------------+-------------------+------------------+------------+
|n_rows |avg_toxicity      |avg_insult         |avg_threat         |avg_forwards      |max_forwards|
+-------+------------------+-------------------+-------------------+------------------+------------+
|1732259|0.3358889519408087|0.20122900212959755|0.13688323743738168|108.38584876741874|98346       |
+-------+------------------+-------------------+-------------------+------------------+------------+



+----------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+------+------+--------+-------------------+
|                                                                                                                                                         content|toxicity|insult|threat|forwards|amplification_score|
+----------------------------------------------------------------------------------------------------------------------------------------------------------------+--------+------+------+--------+-------------------+
|There is going to be a BIG biblical scenario where they make out it's WW3 but really they are activating Militaries then bombing all of these Satanic Lucerif...|     0.4|  0.28|  0.18|   98346| 11.496257320047444|
|There is going to be a BIG biblical scenario where they make out it's WW3 but really they are activating Militaries then bombing all of the

In [9]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, CountVectorizer
from pyspark.ml.clustering import LDA

if FINAL.exists() and list(FINAL.rglob("*.parquet")):
    df_final = spark.read.parquet(str(FINAL))

    tokenizer = RegexTokenizer(inputCol="content", outputCol="tokens", pattern="\\W+", toLowercase=True)
    remover = StopWordsRemover(inputCol="tokens", outputCol="filtered")
    vectorizer = CountVectorizer(inputCol="filtered", outputCol="features", vocabSize=30000, minDF=10)
    lda = LDA(k=20, maxIter=15, featuresCol="features", seed=42)

    pipeline = Pipeline(stages=[tokenizer, remover, vectorizer, lda])
    lda_model_pipeline = pipeline.fit(df_final)

    print("LDA training complete.")

    cv_model = lda_model_pipeline.stages[2]
    lda_model = lda_model_pipeline.stages[3]
    vocab = cv_model.vocabulary
    topics = lda_model.describeTopics(10).collect()

    for row in topics:
        terms = [vocab[i] for i in row["termIndices"]]
        print(f"Topic {row['topic']}: {', '.join(terms)}")

LDA training complete.
Topic 0: child, children, epstein, sex, u, clinton, lied, trafficking, new, sexual
Topic 1: jews, israel, jewish, zionist, america, people, us, world, re, government
Topic 2: police, breaking, muslim, illegal, irish, man, migrants, people, uk, arrested
Topic 3: 2024, 1, 2, trump, us, covid, 3, 11, 10, one
Topic 4: gaza, israeli, al, occupation, lebanon, israel, zionist, channel, hezbollah, us
Topic 5: us, military, world, china, 000, 11, war, event, 1, 9
Topic 6: trump, join, people, re, us, biden, know, world, one, truth
Topic 7: world, border, musk, people, illegal, com, elon, war, immigration, country
Topic 8: trump, us, one, god, people, like, israel, time, get, never
Topic 9: al, zionist, israeli, enemy, brigades, gaza, forces, soldiers, targeted, resistance
Topic 10: people, world, us, one, like, want, government, many, even, war
Topic 11: military, trump, world, cia, operations, deep, state, u, white, going
Topic 12: jews, jewish, hitler, jew, de, israel, 

In [10]:
from pyspark.sql.functions import udf
from pyspark.sql.types import IntegerType
from pyspark.ml.clustering import KMeans

if 'lda_model_pipeline' in globals():
    def argmax_topic(v):
        arr = v.toArray().tolist()
        return int(max(range(len(arr)), key=lambda i: arr[i]))

    argmax_udf = udf(argmax_topic, IntegerType())

    df_topic = lda_model_pipeline.transform(df_final)
    df_topic = df_topic.withColumn("dominant_topic", argmax_udf(col("topicDistribution")))

    kmeans = KMeans(k=6, featuresCol="topicDistribution", predictionCol="cluster", seed=42)
    kmeans_model = kmeans.fit(df_topic)
    df_cluster = kmeans_model.transform(df_topic)

    df_cluster.groupBy("cluster").agg(
        count("*").alias("n_rows"),
        avg("toxicity").alias("avg_toxicity"),
        avg("forwards").alias("avg_forwards"),
        spark_max("forwards").alias("max_forwards")
    ).orderBy("n_rows", ascending=False).show(truncate=False)

+-------+------+-------------------+------------------+------------+
|cluster|n_rows|avg_toxicity       |avg_forwards      |max_forwards|
+-------+------+-------------------+------------------+------------+
|5      |910318|0.3149756678435562 |108.26171733394264|98346       |
|1      |251711|0.37383109995192887|90.48636730218385 |8182        |
|0      |199903|0.3577527600886422 |126.82620570976924|25739       |
|3      |177462|0.34522241381253493|80.71539822609911 |15711       |
|2      |133264|0.35612168327530364|140.39140352983551|41515       |
|4      |59601 |0.3487077398030242 |134.85315682622775|8461        |
+-------+------+-------------------+------------------+------------+

